# DE3 - Train/Test Split

## Load Cleaned Data

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/cleaned_retail.csv",
    parse_dates=["InvoiceDate"]
)

print(df.shape)
print(df["InvoiceDate"].min())
print(df["InvoiceDate"].max())

(802965, 8)
2009-12-01 07:45:00
2011-12-09 12:50:00


## Define the Test Period


In [2]:
max_date = df["InvoiceDate"].max()
cutoff_date = max_date - pd.DateOffset(months=2)

print("Latest date:", max_date)
print("Test period starts:", cutoff_date)

Latest date: 2011-12-09 12:50:00
Test period starts: 2011-10-09 12:50:00


## Create Training and Test Sets

In [3]:
train = df[df["InvoiceDate"] < cutoff_date].copy()
test = df[df["InvoiceDate"] >= cutoff_date].copy()

print("Training rows:", len(train))
print("Test rows:", len(test))

Training rows: 684357
Test rows: 118608


## Keep Only New Products in the Test Set

In [4]:
train_pairs = set(
    zip(train["Customer ID"], train["StockCode"])
)

before = len(test)

test_new = test[
    ~test.apply(
        lambda row: (row["Customer ID"], row["StockCode"]) in train_pairs,
        axis=1
    )
].copy()

print("Test rows before holdout filter:", before)
print("New-product test rows:", len(test_new))
print("Rows removed because product was already seen:", before - len(test_new))

Test rows before holdout filter: 118608
New-product test rows: 72924
Rows removed because product was already seen: 45684


## Save Train and Test Data

In [5]:
train.to_csv("../data/processed/train.csv", index=False)
test_new.to_csv("../data/processed/test_new.csv", index=False)

print("Training and test datasets saved.")

Training and test datasets saved.


# DE4 - Customer-Product Matrix

## Build the Training Matrix

In [6]:
import numpy as np
import scipy.sparse as sp

customer_ids = train["Customer ID"].unique()
product_codes = train["StockCode"].unique()

customer_map = {customer: i for i, customer in enumerate(customer_ids)}
product_map = {product: j for j, product in enumerate(product_codes)}

rows = train["Customer ID"].map(customer_map)
cols = train["StockCode"].map(product_map)
values = np.log1p(train["Quantity"])

A = sp.csr_matrix(
    (values, (rows, cols)),
    shape=(len(customer_map), len(product_map))
)

print("Matrix shape:", A.shape)
print("Non-zero entries:", A.nnz)

Matrix shape: (5484, 4537)
Non-zero entries: 417809


# DE5 - Matrix Report

In [7]:
dense_memory_mb = (
    A.shape[0] * A.shape[1] * 8
) / (1024 ** 2)

sparse_memory_mb = (
    A.data.nbytes
    + A.indices.nbytes
    + A.indptr.nbytes
) / (1024 ** 2)

sparsity = 1 - (
    A.nnz / (A.shape[0] * A.shape[1])
)

print("Matrix shape:", A.shape)
print("Non-zero entries:", A.nnz)
print(f"Sparsity: {sparsity:.2%}")
print(f"Dense memory: {dense_memory_mb:.2f} MB")
print(f"Sparse memory: {sparse_memory_mb:.2f} MB")

Matrix shape: (5484, 4537)
Non-zero entries: 417809
Sparsity: 98.32%
Dense memory: 189.83 MB
Sparse memory: 4.80 MB


### Interpretation

The matrix is extremely sparse, which means most customers purchase only a small fraction of the available products. Therefore, Truncated SVD is a better fit for the main model because it can work directly with the sparse matrix without first converting it into a dense representation.